In [1]:
# put this at the top of every notebook
import plotly.io as pio
pio.renderers.default = "notebook_connected"
#pio.renderers.default = "browser"   # always opens in browser, no nbformat needed


In [2]:
# Cell 1 — setup
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics
from yfinance_api3.classes.options import OptionsAnalyzer
import yfinance_api3.modules.plots as plots

client = StockClient()
quant  = QuantAnalytics(client)
opt    = OptionsAnalyzer(client, "SPY")

In [3]:
# Cell 2 — expiries
print(opt.expiries())


['2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08', '2026-05-15', '2026-05-22', '2026-05-29', '2026-06-05', '2026-06-18', '2026-06-30', '2026-07-17', '2026-07-31', '2026-08-21', '2026-08-31', '2026-09-18', '2026-09-30', '2026-12-18', '2026-12-31', '2027-01-15', '2027-03-19', '2027-03-31', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-06-16', '2028-12-15']


In [4]:
# Cell 3 — front month chain
expiry = opt.nearest_expiry(0)
print(f"Front month: {expiry}")
df = opt.chain(expiry)
print(df.head(20))

Front month: 2026-05-01
        expiry  type  strike  bid  ask  last_price  volume  open_interest  \
0   2026-05-01  call   400.0  0.0  0.0      317.13    11.0            153   
1   2026-05-01   put   400.0  0.0  0.0        0.01   136.0              0   
2   2026-05-01   put   405.0  0.0  0.0        0.03   110.0              0   
3   2026-05-01  call   405.0  0.0  0.0      305.21   296.0            116   
4   2026-05-01  call   410.0  0.0  0.0      304.07     2.0              1   
5   2026-05-01   put   410.0  0.0  0.0        0.01   200.0              0   
6   2026-05-01   put   415.0  0.0  0.0        0.01    11.0              0   
7   2026-05-01  call   415.0  0.0  0.0      299.14    12.0              2   
8   2026-05-01  call   420.0  0.0  0.0      294.16    10.0              2   
9   2026-05-01   put   420.0  0.0  0.0        0.01    10.0           1696   
10  2026-05-01   put   425.0  0.0  0.0        0.01    10.0            159   
11  2026-05-01   put   430.0  0.0  0.0        0.01  

In [5]:
# Cell 4 — summary
print(opt.summary(expiry))

{'symbol': 'SPY', 'expiry': '2026-05-01', 'days_to_expiry': 0, 'n_strikes': 213, 'total_call_oi': 379900, 'total_put_oi': 205105, 'total_call_vol': 756994, 'total_put_vol': 887010, 'put_call_ratio_oi': 0.54, 'put_call_ratio_vol': 1.172, 'max_pain_strike': 700.0, 'max_oi_call_strike': np.float64(708.0), 'max_oi_put_strike': np.float64(690.0), 'avg_call_iv': 0.0627, 'avg_put_iv': 0.3302, 'iv_skew': 0.2675}


In [6]:
# Cell 5 — max pain (knockout price)
print(f"Max pain: ${opt.max_pain(expiry):,.2f}")

Max pain: $700.00


In [7]:
# Cell 6 — put/call ratio
print(opt.put_call_ratio())

        expiry  days_to_expiry  call_oi   put_oi  put_call_ratio
0   2026-05-01               0   379900   205105           0.540
1   2026-05-04               3    62913   151945           2.415
2   2026-05-05               4    17345    16684           0.962
3   2026-05-06               5     3882    10073           2.595
4   2026-05-07               6     9166     5574           0.608
5   2026-05-08               7    35678   129185           3.621
6   2026-05-15              14   726062  1854318           2.554
7   2026-05-22              21    92129   533164           5.787
8   2026-05-29              28   229530  1290645           5.623
9   2026-06-05              35    18012    32269           1.792
10  2026-06-18              48   831692  1941389           2.334
11  2026-06-30              60   158802   301518           1.899
12  2026-07-17              77   174294   299620           1.719
13  2026-07-31              91    82616   139049           1.683
14  2026-08-21           

In [8]:
plots.options_chain(opt, opt.nearest_expiry(1)).show()

In [9]:
plots.options_oi_profile(opt, opt.nearest_expiry(1)).show()

In [10]:
plots.options_put_call(opt).show()

In [11]:
plots.options_surface(opt, max_expiries=6).show()


In [12]:
# front 12 expiries (~1 year for weeklies, or full year for monthlies)
plots.options_max_pain(opt, max_expiries=12).show()



In [13]:
# more expiries
plots.options_max_pain(opt, max_expiries=24).show()

In [14]:
plots.options_gex(opt, max_expiries=12).show()


In [15]:
plots.options_unusual(opt, vol_oi_threshold=3.0, min_volume=100).show()

In [16]:
df = opt.greeks(opt.nearest_expiry(0))
print("Greeks columns:", df.columns.tolist())

Greeks columns: ['expiry', 'type', 'strike', 'bid', 'ask', 'last_price', 'volume', 'open_interest', 'implied_volatility', 'in_the_money', 'contract', 'delta', 'gamma', 'theta', 'vega', 'rho', 'theoretical_price', 'greek_source']


In [17]:
#opt = OptionsAnalyzer(client, "SPY")

# Greeks

print(df[["type","strike","delta","gamma","theta","vega","rho","greek_source"]].head(10))


   type  strike  delta  gamma   theta  vega     rho   greek_source
0  call   400.0    1.0    0.0 -0.0548   0.0  0.0110  black_scholes
1   put   400.0    0.0    0.0 -0.0000   0.0 -0.0000  black_scholes
2   put   405.0    0.0    0.0 -0.0000   0.0 -0.0000  black_scholes
3  call   405.0    1.0    0.0 -0.0555   0.0  0.0111  black_scholes
4  call   410.0    1.0    0.0 -0.0562   0.0  0.0112  black_scholes
5   put   410.0    0.0    0.0 -0.0000   0.0 -0.0000  black_scholes
6   put   415.0    0.0    0.0 -0.0000   0.0 -0.0000  black_scholes
7  call   415.0    1.0    0.0 -0.0568   0.0  0.0114  black_scholes
8  call   420.0    1.0    0.0 -0.0575   0.0  0.0115  black_scholes
9   put   420.0    0.0    0.0 -0.0000   0.0 -0.0000  black_scholes


In [18]:
# GEX
strike_gex = opt.gex_by_strike()
by_exp     = opt.gex_by_expiry()
total      = opt.gex_total()

flip     = total.get("flip_strike")
flip_str = f"${flip:,.2f}" if flip is not None else "none detected"
print(f"\nRegime:          {total['regime_label']}")
print(f"Total GEX:       ${total['total_net_gex']/1e6:.1f}M")
print(f"GEX flip:        {flip_str}")
print(f"Dominant expiry: {total.get('dominant_expiry', '—')}")
# Unusual activity
unusual = opt.unusual_activity(vol_oi_threshold=3.0, min_volume=100)
print(f"\nUnusual activity — {len(unusual)} strikes flagged")
if not unusual.empty:
    print(unusual[["type","strike","expiry","volume",
                   "open_interest","vol_oi_ratio","signal"]].head(10))


Regime:          Positive GEX — market makers are net long gamma. They sell rallies and buy dips → price-stabilising, expect lower realised volatility.
Total GEX:       $28074.1M
GEX flip:        $400.00
Dominant expiry: 2026-05-15

Unusual activity — 87 strikes flagged
   type  strike      expiry   volume  open_interest  vol_oi_ratio     signal
0  call   713.0  2026-05-01  30074.0           2767         10.87  🚨 Unusual
1  call   714.0  2026-05-01  35397.0           4337          8.16  🚨 Unusual
2  call   721.0  2026-05-01  37749.0           4932          7.65  🚨 Unusual
3  call   719.0  2026-05-01  61254.0           8083          7.58  🚨 Unusual
4  call   716.0  2026-05-01  54454.0           7335          7.42  🚨 Unusual
5  call   718.0  2026-05-01  55010.0           7820          7.03  🚨 Unusual
6   put   713.0  2026-05-01  39733.0           6019          6.60  🚨 Unusual
7   put   711.0  2026-05-01  38413.0           6353          6.05  🚨 Unusual
8  call   715.0  2026-05-01  90283.

In [19]:
total = opt.gex_total()
print(total.keys())

dict_keys(['symbol', 'total_call_gex', 'total_put_gex', 'total_net_gex', 'regime', 'regime_label', 'flip_strike', 'dominant_expiry'])


In [20]:
by_exp = opt.gex_by_expiry()
print(by_exp.columns.tolist())
print(by_exp.head(2))

['expiry', 'days_to_expiry', 'call_gex', 'put_gex', 'net_gex', 'abs_gex']
       expiry  days_to_expiry      call_gex       put_gex       net_gex  \
0  2026-05-01               0  2.988291e+09 -4.020168e+06  2.984271e+09   
1  2026-05-04               3  3.129104e+09 -1.148832e+06  3.127956e+09   

        abs_gex  
0  2.992311e+09  
1  3.130253e+09  


In [21]:
from yfinance_api3.classes.pricing import (
    ContractType, Space, ExerciseType
)
from yfinance_api3.classes.positions_book import (
    PositionsBook, PortfolioBook, WatchList
)

# ── Build INTC book ───────────────────────────────────────────────────────
book = PositionsBook(
    symbol         = "INTC",
    spot           = 93.22,
    vol            = 0.61,           # 61% hist vol
    risk_free_rate = 4.50,           # 4.50%
    div_yield      = 0.0,
    contract_type  = ContractType.STOCK,
    lot_size       = 100,
)

# add legs from your sheet (negative lots = short)
book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61, "equity")
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61, "equity")
book.add_option("call", "2026-01-30", 40,   0,  1.80, 0.54, "equity")
book.add_option("call", "2026-06-18", 40,   0,  4.50, 0.61, "equity")
book.add_option("call", "2026-06-18", 50,   0,  7.30, 0.61, "equity")
book.add_option("call", "2026-01-30", 37,   0, 10.40, 0.52, "equity")

# underlying position
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# ── Greeks table ──────────────────────────────────────────────────────────
print(book.greeks_table(days_ahead=0).to_string())

# ── Summary ───────────────────────────────────────────────────────────────
s = book.summary(days_ahead=0)
for k, v in s.items():
    print(f"  {k:20s}: {v}")

# ── +30 days simulation ───────────────────────────────────────────────────
print("\n--- +30 days ---")
s30 = book.summary(days_ahead=30)
for k, v in s30.items():
    print(f"  {k:20s}: {v}")

# ── Save ─────────────────────────────────────────────────────────────────
book.save("positions/INTC.json")
print("\nSaved to positions/INTC.json")

# ── Portfolio ─────────────────────────────────────────────────────────────
port = PortfolioBook(name="My portfolio")
port.add_book(book, path="positions/INTC.json")
print(port.summary())
port.save("positions/portfolio.json")

# ── WatchList ─────────────────────────────────────────────────────────────
wl = WatchList()
wl.add("INTC", notes="short calls position")
wl.add("SPY",  notes="hedge")
wl.save("positions/watchlist.json")

     leg_id         type      expiry strike     iv          model exercise   lots lot_size price_paid model_price      value        pnl     delta  gamma     vega   theta      rho
0  986deed2         Call  2026-12-18   85.0  61.0%  Black-Scholes        E    -60      100      18.11     22.6668 -136000.62  -27340.62  -4130.89 -46.89 -1573.12  238.42 -1576.37
1  cfe818c7         Call  2026-12-18   90.0  61.0%  Black-Scholes        E    -40      100       21.5       20.34  -81359.84    4640.16  -2582.84 -32.90 -1103.58  165.36 -1008.88
2  b4494920         Call  2026-01-30   40.0  54.0%  Black-Scholes        E      0      100        1.8       53.22       0.00       0.00      0.00   0.00     0.00    0.00     0.00
3  66e32016         Call  2026-06-18   40.0  61.0%  Black-Scholes        E      0      100        4.5     53.4562       0.00       0.00      0.00   0.00     0.00   -0.00     0.00
4  44a124d5         Call  2026-06-18   50.0  61.0%  Black-Scholes        E      0      100        7.3    

In [22]:
from yfinance_api3.classes.pricing import Space, ContractType
from yfinance_api3.classes.positions_book import PositionsBook, PortfolioBook

# rebuild INTC book
book = PositionsBook(
    symbol="INTC", spot=93.22, vol=0.61,
    risk_free_rate=4.50, div_yield=0.0,
)
book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61)
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61)
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# single ticker dashboard — replicates your sheet
#plots.positions_book(book, days_ahead=0).show()
#plots.positions_book(book, days_ahead=30).show()   # +30 days simulation

plots.positions_book(book, days_ahead=30).show()   # chart + summary


In [23]:
plots.positions_legs(book, days_ahead=30).show()   # legs table



In [24]:
# portfolio summary
port = PortfolioBook(name="My portfolio")
port.add_book(book)
plots.portfolio_summary(port, days_ahead=0).show()

In [25]:
# Debug GEX
gex = opt.gex_by_strike()
print("GEX sample:")
print(gex.head(5))
print(f"\nMax abs net_gex: {gex['net_gex'].abs().max():.2f}")
print(f"Non-zero rows: {(gex['net_gex'] != 0).sum()} / {len(gex)}")

# Check gamma values
df = opt.greeks(opt.nearest_expiry(0))
print(f"\nGamma stats:")
print(df["gamma"].describe())
print(f"greek_source counts: {df['greek_source'].value_counts().to_dict()}")

GEX sample:
   strike  call_gex  put_gex  net_gex  cumulative_gex
0   400.0       0.0     -0.0      0.0             0.0
1   405.0       0.0     -0.0      0.0             0.0
2   410.0       0.0     -0.0      0.0             0.0
3   415.0       0.0     -0.0      0.0             0.0
4   420.0       0.0     -0.0      0.0             0.0

Max abs net_gex: 2949695224.10
Non-zero rows: 44 / 213

Gamma stats:
count    382.000000
mean       0.001919
std        0.036153
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.706574
Name: gamma, dtype: float64
greek_source counts: {'black_scholes': 382}


In [26]:
df = opt.greeks(opt.nearest_expiry(0))
print(df[["type","strike","implied_volatility","gamma","greek_source"]].head(10))
print(f"\nIV stats: min={df['implied_volatility'].min()}, max={df['implied_volatility'].max()}")
print(f"Zero IV rows: {(df['implied_volatility'] == 0).sum()} / {len(df)}")

   type  strike  implied_volatility  gamma   greek_source
0  call   400.0            0.000010    0.0  black_scholes
1   put   400.0            0.500005    0.0  black_scholes
2   put   405.0            0.500005    0.0  black_scholes
3  call   405.0            0.000010    0.0  black_scholes
4  call   410.0            0.000010    0.0  black_scholes
5   put   410.0            0.500005    0.0  black_scholes
6   put   415.0            0.500005    0.0  black_scholes
7  call   415.0            0.000010    0.0  black_scholes
8  call   420.0            0.000010    0.0  black_scholes
9   put   420.0            0.500005    0.0  black_scholes

IV stats: min=1.0000000000000003e-05, max=2.576663714599609
Zero IV rows: 0 / 382
